# Recodificación Pillow

Metodo para recodificar imagenes de formatos usando la libreria de PILLOW

In [1]:
from PIL import Image
import os

def convert_image_pillow(input_path, output_path, quality=90):
    img = Image.open(input_path)

    # Asegurar conversión correcta si tiene transparencia
    if img.mode in ("RGBA", "LA", 'P'):
        img = img.convert("RGB")

    img.save(output_path, quality=quality)
    print(f"Convertido: {input_path} → {output_path}")


# Recodificación ImageMagick

Metodo para recodificar imagenes en los diferentes formatos usando la libreria wand 

In [5]:
from wand.image import Image as WandImage
import os

def convert_with_imagemagick(
    input_path,
    output_path,
    quality=90
):
    formato = os.path.splitext(output_path)[1].replace(".", "").lower()

    with WandImage(filename=input_path) as img:

        # ✅ Forzar conversión a RGB si es escala de grises
        if img.colorspace in ('gray', 'grey', 'linear_gray'):
            img.transform_colorspace('srgb')
        
        img.colorspace = "srgb"
        img.type = "truecolor"

        img.compression_quality = quality
        img.format = formato

        if formato == "png":
            img.options["png:color-type"] = "2"

        img.save(filename=output_path)

    print(f"[ImageMagick] Convertido → {output_path}")

# Recodificación FFmpeg

Metodo para recodificar imagenes en los diferentes formatos usando FFmpeg

In [1]:
import subprocess

def convert_with_ffmpeg(input_path, output_path, quality=90):
    
    formato = output_path.rsplit(".", 1)[-1].lower()
    
    # Mapeo correcto: quality 0-100 → ffmpeg 31-2 (invertido)
    ffmpeg_quality = max(2, min(31, int(31 - (quality / 100) * 29)))
    
    if formato == "webp":
        cmd = [
            "ffmpeg",
            "-y",
            "-i", input_path,
            "-quality", str(quality),  # webp usa 0-100 directo
            output_path
        ]
    else:
        cmd = [
            "ffmpeg",
            "-y",
            "-i", input_path,
            "-q:v", str(ffmpeg_quality),  # jpg usa escala 2-31
            output_path
        ]
    
    result = subprocess.run(
        cmd, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.PIPE
    )
    
    # ✅ Verificar si hubo error
    if result.returncode != 0:
        print(f"[FFmpeg] ❌ Error en {input_path}")
        print(result.stderr.decode())
        return False
        
    print(f"[FFmpeg] Convertido → {output_path}")
    return True